# Building an AI Agent with Tools

In this exercise, you'll build an AI agent that can use tools to enhance its capabilities. You'll learn how to create an agent that can understand when to use tools, process their results, and maintain a coherent conversation.

## Challenge

Imagine you're building a smart coding assistant that needs to:
- Answer programming questions
- Execute code snippets
- Look up documentation
- Perform calculations
- Search through codebases

Instead of hard-coding when to use each capability, your agent should intelligently decide when and how to use its available tools.

## Setup
First, let's import the necessary libraries:

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"

In [4]:
from typing import List, Dict, Any
from dotenv import load_dotenv
from copy import deepcopy
import json

from lib.messages import UserMessage, SystemMessage, ToolMessage
from lib.tooling import tool
from lib.llm import LLM

## Understanding the Components

Before we build our agent, let's understand the key components we'll be working with:

- `LLM`: The language model wrapper that handles tool execution
- `SystemMessage`: Defines the agent's role and behavior
- `UserMessage`: Represents user inputs
- `ToolMessage`: Contains tool execution results
- `tool`: Decorator for creating tools

## Building the Agent Class

Your task is to create an Agent class that can:
1. Initialize with a specific role and set of tools
2. Process user messages
3. Decide when to use tools
4. Handle tool responses

In [21]:
class Agent:
    """An AI Agent that can use tools to help answer questions"""
    
    def __init__(
        self,
        role: str = "Personal Assistant",
        instructions: str = "Help users with any question",
        model: str = "gpt-4o-mini",
        temperature: float = 0.0,
        tools: List[Any] = None
    ):
        """Initialize the agent with its configuration and tools"""
        load_dotenv()

        self.role = role
        self.instructions = instructions
        self.model = model
        self.temperature = temperature
        self.tools = tools

        self.llm = LLM(
            model=model,
            temperature=temperature,
            tools=tools
        )

    def invoke(self, user_message: str) -> str:
        """Process a user message and return a response"""
        messages:  list = [
            SystemMessage(
                content=(
                    f"You are an AI agent and your role is {self.role}. "
                    f"Your instruction is {self.instructions}"
                )
            )
        ]
        messages.append(UserMessage(content=user_message))
        ai_message = self.llm.invoke(messages)
        messages.append(ai_message)

        # check whether tools are required
        while ai_message.tool_calls:
            for call in ai_message.tool_calls:
                function_name = call.function.name
                function_args = json.loads(call.function.arguments)
                tool_call_id = call.id
                # find the matching tool call
                tool = next((t for t in self.tools if t.name == function_name), None)
                if tool:
                    result = tool(**function_args)
                    messages.append(
                        ToolMessage(
                            content=json.dumps(result),
                            tool_call_id=tool_call_id,
                            name=function_name
                        )
                    )
            ai_message = self.llm.invoke(messages)
            messages.append(ai_message)
        for m in messages:
            print(m)
        return ai_message.content

## Testing Your Agent

Once you've implemented the Agent class, test it with different scenarios:

1. Basic conversation without tools

In [8]:
agent = Agent(role="Coding Assistant")
response = agent.invoke("What is Python? Be concise")
print(response)

role='system' content='You are an AI agent and your role is Coding Assistant. Your instruction is Help users with any question'
role='user' content='What is Python? Be concise'
role='assistant' content='Python is a high-level, interpreted programming language known for its readability and simplicity. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python is widely used for web development, data analysis, artificial intelligence, scientific computing, and automation, among other applications.' tool_calls=None token_usage=TokenUsage(prompt_tokens=37, completion_tokens=57, total_tokens=94)
Python is a high-level, interpreted programming language known for its readability and simplicity. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python is widely used for web development, data analysis, artificial intelligence, scientific computing, and automation, among other 

2. Create a calculator tool

In [18]:
@tool
def calculate(expression: str) -> float:
    """Evaluate a mathematical expression"""
    return eval(expression)

In [22]:
# Create an agent with the calculator tool
math_agent = Agent(
    role="Math Assistant",
    tools=[calculate]
)

In [23]:
response = math_agent.invoke("What is 23 * 45?")
print(response)

role='system' content='You are an AI agent and your role is Math Assistant. Your instruction is Help users with any question'
role='user' content='What is 23 * 45?'
role='assistant' content=None tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_9zqEoif08MlbfxTrhBZ69r1e', function=Function(arguments='{"expression":"23 * 45"}', name='calculate'), type='function')] token_usage=TokenUsage(prompt_tokens=70, completion_tokens=16, total_tokens=86)
role='tool' content='1035' tool_call_id='call_9zqEoif08MlbfxTrhBZ69r1e' name='calculate'
role='assistant' content='The result of \\( 23 \\times 45 \\) is 1035.' tool_calls=None token_usage=TokenUsage(prompt_tokens=95, completion_tokens=18, total_tokens=113)
The result of \( 23 \times 45 \) is 1035.


In [24]:
# Test multiple tool usage
response = math_agent.invoke("If I multiply 3 by 5, what do I get? Then later add 7")
print(response)

role='system' content='You are an AI agent and your role is Math Assistant. Your instruction is Help users with any question'
role='user' content='If I multiply 3 by 5, what do I get? Then later add 7'
role='assistant' content=None tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_uz9g6f64B412oIkGUiisYRQb', function=Function(arguments='{"expression": "3 * 5"}', name='calculate'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_7MiyZdYBEhSa4ngmdAPkxVUO', function=Function(arguments='{"expression": "3 * 5 + 7"}', name='calculate'), type='function')] token_usage=TokenUsage(prompt_tokens=81, completion_tokens=51, total_tokens=132)
role='tool' content='15' tool_call_id='call_uz9g6f64B412oIkGUiisYRQb' name='calculate'
role='tool' content='22' tool_call_id='call_7MiyZdYBEhSa4ngmdAPkxVUO' name='calculate'
role='assistant' content='If you multiply 3 by 5, you get 15. If you then add 7 to that result, you get 22.' tool_calls=None token_usage=TokenUsage(prompt_tokens=148

3. Create a data analyst

In [25]:
@tool
def get_games(num_games:int=1, top:bool=True) -> str:
    """
    Returns the top or bottom N games with highest or lowest scores.    
    args:
        num_games (int): Number of games to return (default is 1)
        top (bool): If True, return top games, otherwise return bottom (default is True)
    """
    data = [
        {"Game": "The Legend of Zelda: Breath of the Wild", "Platform": "Switch", "Score": 98},
        {"Game": "Super Mario Odyssey", "Platform": "Switch", "Score": 97},
        {"Game": "Metroid Prime", "Platform": "GameCube", "Score": 97},
        {"Game": "Super Smash Bros. Brawl", "Platform": "Wii", "Score": 93},
        {"Game": "Mario Kart 8 Deluxe", "Platform": "Switch", "Score": 92},
        {"Game": "Fire Emblem: Awakening", "Platform": "3DS", "Score": 92},
        {"Game": "Donkey Kong Country Returns", "Platform": "Wii", "Score": 87},
        {"Game": "Luigi's Mansion 3", "Platform": "Switch", "Score": 86},
        {"Game": "Pikmin 3", "Platform": "Wii U", "Score": 85},
        {"Game": "Animal Crossing: New Leaf", "Platform": "3DS", "Score": 88}
    ]
    # Sort the games list by Score
    # If top is True, descending order
    sorted_games = sorted(data, key=lambda x: x['Score'], reverse=top)
    
    # Return the N games
    return sorted_games[:num_games]

In [27]:
# Create an agent with the multiple tools
data_analyst_agent = Agent(
    role="Game Stats Assistant",
    instructions="You can bring insights about a game dataset based on users questions",
    tools=[get_games]
)

In [28]:
response = data_analyst_agent.invoke("What's the best game in the dataset?")
print(response)

role='system' content='You are an AI agent and your role is Game Stats Assistant. Your instruction is You can bring insights about a game dataset based on users questions'
role='user' content="What's the best game in the dataset?"
role='assistant' content=None tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_HykAygtkb2XPzGL3m5pXv53q', function=Function(arguments='{"num_games":1,"top":true}', name='get_games'), type='function')] token_usage=TokenUsage(prompt_tokens=129, completion_tokens=19, total_tokens=148)
role='tool' content='[{"Game": "The Legend of Zelda: Breath of the Wild", "Platform": "Switch", "Score": 98}]' tool_call_id='call_HykAygtkb2XPzGL3m5pXv53q' name='get_games'
role='assistant' content='The best game in the dataset is "The Legend of Zelda: Breath of the Wild" for the Switch, with a score of 98.' tool_calls=None token_usage=TokenUsage(prompt_tokens=183, completion_tokens=30, total_tokens=213)
The best game in the dataset is "The Legend of Zelda: Breath of the 

In [29]:
response = data_analyst_agent.invoke("What's the worst game in the dataset?")
print(response)

role='system' content='You are an AI agent and your role is Game Stats Assistant. Your instruction is You can bring insights about a game dataset based on users questions'
role='user' content="What's the worst game in the dataset?"
role='assistant' content=None tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_LgNotujfHYL2Pow18vs2qfsS', function=Function(arguments='{"num_games":1,"top":false}', name='get_games'), type='function')] token_usage=TokenUsage(prompt_tokens=129, completion_tokens=19, total_tokens=148)
role='tool' content='[{"Game": "Pikmin 3", "Platform": "Wii U", "Score": 85}]' tool_call_id='call_LgNotujfHYL2Pow18vs2qfsS' name='get_games'
role='assistant' content='The worst game in the dataset is "Pikmin 3" for the Wii U, with a score of 85.' tool_calls=None token_usage=TokenUsage(prompt_tokens=181, completion_tokens=27, total_tokens=208)
The worst game in the dataset is "Pikmin 3" for the Wii U, with a score of 85.


In [2]:
game_stats_agent = Agent(
    role="Game Stats Assistant",
    instructions="You can bring insights about a game dataset based on users questions",
    tools=[get_games, calculate]
)

NameError: name 'Agent' is not defined

In [34]:
response = game_stats_agent.invoke("What is the average score?")
print(response)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (<string>, line 1)

In [ ]:
response = game_stats_agent.invoke("What is the distribution of platforms?")
print(response)